In [1]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="m3rg-iitd/matscibert", local_dir="./matscibert")

/Users/siyuliu/anaconda3/envs/bmg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:01<00:00,  6.77it/s]


'/Users/siyuliu/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/Project/BMG/MgBERT_LLM_Classification_for_Materials_Science/matscibert'

In [2]:
from tokenizers.normalizers import BertNormalizer


f = open('vocab_mappings.txt', 'r')
mappings = f.read().strip().split('\n')
f.close()

mappings = {m[0]: m[2:] for m in mappings}

norm = BertNormalizer(lowercase=False, strip_accents=True, clean_text=True, handle_chinese_chars=True)

def normalize(text):
    text = [norm.normalize_str(s) for s in text.split('\n')]
    out = []
    for s in text:
        norm_s = ''
        for c in s:
            norm_s += mappings.get(c, ' ')
        out.append(norm_s)
    return '\n'.join(out)

In [3]:
import torch
from torch import nn
import random
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Subset
from transformers import AutoModel, AutoTokenizer, AutoConfig


def setup_seed(seed):
     torch.manual_seed(seed)
     torch.cuda.manual_seed_all(seed)
     np.random.seed(seed)
     random.seed(seed)
## 设置随机数种子
setup_seed(42)

config = AutoConfig.from_pretrained('./matscibert')
config.max_position_embeddings = 900
bert_model = AutoModel.from_pretrained('./matscibert', config=config, ignore_mismatched_sizes=True)


class BertClassifier(nn.Module):
    def __init__(self, dropout=0.5):
        super(BertClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(768, 3)
        self.relu = nn.ReLU()

    def forward(self, input_id, mask):
        outputs = self.bert(input_ids=input_id, attention_mask=mask,return_dict=True, output_attentions=True)
        pooled_output = outputs.pooler_output
        attentions = outputs.attentions
        dropout_output = self.dropout(pooled_output)
        linear_output = self.linear(dropout_output)
        final_layer = self.relu(linear_output)
        return final_layer, attentions


## 数据获取
tokenizer = AutoTokenizer.from_pretrained('./matscibert')
def find_text():
    file_path = os.path.join('./test.txt')
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found.")
    with open(file_path, 'r') as file:
        text = file.read()
    return text



use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

## 模型读取
from torch.serialization import load
model_path = 'MgBERT.pth'
model_data = torch.load(model_path, map_location=device)
model = BertClassifier()
model.to(device)
model.load_state_dict(model_data)
model.eval()

Some weights of BertModel were not initialized from the model checkpoint at ./matscibert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at ./matscibert and are newly initialized because the shapes did not match:
- bert.embeddings.position_embeddings.weight: found shape torch.Size([512, 768]) in the checkpoint and torch.Size([900, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(900, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

In [4]:
def inference():
    input_text = find_text()
    inputs = tokenizer(normalize(input_text),
                                padding='max_length', 
                                max_length = 900, 
                                truncation=True,
                                return_tensors="pt").to(device)
    output, attention = model(inputs['input_ids'], inputs['attention_mask'])
    return output.argmax(dim=1)

In [5]:
result = inference()
if result == 0:
    print("The composition Mg59.5Cu22.9Ag6.6Gd11 is a bulk metallic glass.")
elif result == 1:
    print("The composition Mg59.5Cu22.9Ag6.6Gd11 is a ribbon-like metallic glass.")
elif result == 2:
    print("The composition Mg59.5Cu22.9Ag6.6Gd11 is not a metallic glass.")

The composition Mg59.5Cu22.9Ag6.6Gd11 is a bulk metallic glass.
